# ML Feature Engineering — Customer-Level Churn Features

Builds a per-customer feature table on top of `fact_sales_enriched` for downstream churn modeling. Output is persisted as a Delta table named `ml_customer_features` with one row per customer and a synthetic `churn_label`.

**Pipeline:**
1. Load `fact_sales_enriched` via `spark.table()`.
2. Pin a reproducible snapshot date from the data itself (not wall-clock).
3. Aggregate 11 features in a single `groupBy(customer_id)` pass.
4. Derive ratio/recency features and the synthetic `churn_label` post-aggregation.
5. Persist as Delta (`mode("overwrite")`) and display.

**Design choices that matter at scale:**

- **Single `groupBy` pass.** Every additional pass over a 100M-row fact costs ~5GB of I/O and a full shuffle on `customer_id`. We compute all features in one aggregation — one scan, one shuffle, many outputs.
- **`approx_count_distinct` over `countDistinct`.** Exact distinct counts require per-group sets that grow with cardinality and can OOM on hot customers. HLL sketches use a fixed ~2KB per group and are accurate to ±5% by default — fine for ML features.
- **Snapshot date from the data, not `current_date()`.** ML features must be reproducible: re-running the pipeline tomorrow on the same input must produce the same features. Wall-clock dates break that contract. We use `max(transaction_date)` over the fact as the reference point.
- **No `collect()` on row data.** The only driver materialization is a single scalar (`max(transaction_date)`) via `.first()` — bounded regardless of fact size.
- **`coalesce(discount_pct, 0)` inside `avg()`.** Treats unpromoted transactions as 0% discount instead of excluding them, producing a single comparable feature across customers regardless of how often they shopped on promotion.
- **Delta `mode("overwrite")` with `overwriteSchema=true`.** Pipeline is re-runnable; schema can evolve as features are added/removed without manual table-drop steps.

In [ ]:
from pyspark.sql import functions as F

# Standalone notebook: assume `spark` is available (Databricks default) but import
# explicitly for OSS-Spark/local-dev runs.
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

## 1. Load `fact_sales_enriched`

`spark.table()` returns a lazy DataFrame backed by the Delta table — no data is read until an action runs. The columnar Parquet scan will only materialize the columns referenced in the `select(...)` below.

In [ ]:
enriched = spark.table("fact_sales_enriched")

# Quick sanity check — schema only, not row data. printSchema() is metadata-only and does
# not trigger a scan.
enriched.printSchema()

## 2. Pin the snapshot date

Reference date for the recency feature. Using `max(transaction_date)` over the fact (rather than `current_date()`) makes the pipeline reproducible: same input → same features, regardless of when it runs.

`.first()` materializes one Row to the driver — bounded cost regardless of fact size, and the only driver materialization in this notebook.

In [ ]:
snapshot_date = enriched.agg(F.max("transaction_date").alias("d")).first()["d"]
print(f"Feature snapshot date: {snapshot_date}")

## 3. Project to ONLY the columns the aggregation reads

Even on a Delta-backed source, an explicit `select()` makes the columnar prune visible in the plan and prevents accidental "carry along everything" if the source schema grows.

In [ ]:
feature_input = enriched.select(
    "customer_id",
    "transaction_date",
    "net_revenue",
    "gross_margin",
    "quantity",
    "discount_pct",
    "category",
    "store_id",
)

## 4. Compute all features in a single `groupBy` pass

One scan of the 100M-row fact, one shuffle on `customer_id`, all aggregations computed in the same partial+final HashAggregate pair. Adding a feature later costs an extra aggregator slot, not an extra pass over the data.

Derived features (`avg_transaction_value`, `days_since_last_purchase`, `churn_label`) are computed AFTER the aggregation, on the small per-customer result — no extra fact scan.

In [ ]:
customer_features = (
    feature_input
    .groupBy("customer_id")
    .agg(
        # Spend volume features.
        F.round(F.sum("net_revenue"), 2).alias("total_revenue"),
        F.count("*").alias("transaction_count"),
        F.sum("quantity").cast("long").alias("total_quantity"),
        F.round(F.sum("gross_margin"), 2).alias("total_gross_margin"),

        # Behavior features. coalesce(discount_pct, 0) treats unpromoted transactions as
        # 0% discount so customers who rarely use promotions get a low avg, not NULL.
        F.round(F.avg(F.coalesce("discount_pct", F.lit(0.0))), 4).alias("avg_discount_pct"),

        # Diversity features — approx_count_distinct uses HyperLogLog with bounded ~2KB/group
        # state. Default rsd=0.05 (5% error). At ~8 categories and 200 stores accuracy is
        # essentially exact; the choice matters for high-cardinality features but is the safe
        # default to avoid OOM on hot customers if cardinality ever grows.
        F.approx_count_distinct("category").alias("distinct_categories"),
        F.approx_count_distinct("store_id").alias("distinct_stores"),

        # Temporal features.
        F.min("transaction_date").alias("first_purchase_date"),
        F.max("transaction_date").alias("last_purchase_date"),
    )
    # Derived: avg = sum/count over the already-aggregated columns. No extra fact scan.
    .withColumn(
        "avg_transaction_value",
        F.round(F.col("total_revenue") / F.col("transaction_count"), 2),
    )
    # Recency feature, anchored to the snapshot_date scalar (broadcast as a literal).
    # datediff is end - start, so this is positive for any customer whose last purchase
    # predates the snapshot.
    .withColumn(
        "days_since_last_purchase",
        F.datediff(F.lit(snapshot_date), F.col("last_purchase_date")),
    )
    # Synthetic churn label. > 60 days since last purchase = churned. Cast to int because
    # downstream ML libraries (MLlib, sklearn-via-pandas-on-Spark) generally expect integer
    # binary labels rather than booleans.
    .withColumn("churn_label", (F.col("days_since_last_purchase") > F.lit(60)).cast("int"))
)

## 5. Persist as Delta

`mode("overwrite")` for re-runnability. `overwriteSchema=true` so this cell tolerates adding/removing features in future iterations without manual table-drop steps.

No partitioning: the table is one row per customer (~200M rows max), and downstream ML code typically wants a full scan rather than partition pruning. If a downstream join back to `dim_customer` is on the critical path, mirror its `bucketBy(32, "customer_id")` layout here.

In [ ]:
(
    customer_features
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ml_customer_features")
)

## 6. Inspect the result

Read back via `spark.table()` rather than reusing the `customer_features` DataFrame so the displays reflect what was actually persisted (catches schema-evolution surprises). Three checks:

1. Schema — confirms all 13 columns landed.
2. Label balance — useful sanity signal for the synthetic churn definition. A 100% / 0% split usually means the synthetic data has no temporal spread, or the threshold needs tuning.
3. Sample rows — quick visual scan for plausibility.

In [ ]:
features_table = spark.table("ml_customer_features")

print("ml_customer_features schema:")
features_table.printSchema()

print("\nLabel distribution:")
display(
    features_table
    .groupBy("churn_label")
    .agg(F.count("*").alias("customers"))
    .orderBy("churn_label")
)

print("\nSample feature rows:")
display(features_table)